# 02. Feature Engineering

We calculate rolling features, lags, and weather proxies (NDVI).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../backend/data/raw/synthetic_prices.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["commodity", "date"])

# Create lags and rolling features
def create_features(group):
    group["price_lag_7"] = group["price"].shift(7)
    group["price_lag_14"] = group["price"].shift(14)
    group["price_lag_30"] = group["price"].shift(30)
    group["rolling_mean_7"] = group["price"].rolling(window=7).mean()
    group["rolling_mean_30"] = group["price"].rolling(window=30).mean()
    group["rolling_std_7"] = group["price"].rolling(window=7).std()
    
    # Calendar features
    group["day_of_week"] = group["date"].dt.dayofweek
    group["month"] = group["date"].dt.month
    
    # Fake Weather Features
    group["temperature"] = 25 + 10 * np.sin(group["date"].dt.dayofyear / 365.0 * 2 * np.pi) + np.random.normal(0, 2, len(group))
    group["rainfall"] = np.where(group["month"].isin([6,7,8,9]), np.random.gamma(2, 5, len(group)), np.random.exponential(1, len(group)))
    
    # NDVI proxy (1 - (30 - rainfall_30d)/30)
    group["rainfall_30d"] = group["rainfall"].rolling(window=30).mean().fillna(0)
    group["ndvi_proxy"] = np.clip(1 - (30 - group["rainfall_30d"]) / 30, 0, 1)
    
    return group

df_features = df.groupby("commodity", group_keys=False).apply(create_features).dropna()
import os
os.makedirs("../backend/data/processed", exist_ok=True)
df_features.to_csv("../backend/data/processed/engineered_features.csv", index=False)
print("Features created. Shape:", df_features.shape)
df_features.head()